## Part 1: Setup & Document Selection 

In [1]:
pip install python-dotenv langsmith openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-custom-dataset-evaluation"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith endpoint loaded:", bool(os.getenv("LANGSMITH_ENDPOINT")))
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))


OpenAI key loaded: True
LangSmith key loaded: True
LangSmith endpoint loaded: True
OpenAI key loaded: True


In [3]:
from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from datasets import load_dataset

client = Client()
openai_client = wrap_openai(OpenAI())

c:\Users\antun\miniconda3\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Test
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Reply with a short greeting and a check emoji at the end.",
        }
    ],
)

answer = response.choices[0].message.content
print(answer)

Hello! Hope you're doing well! ✅


In [9]:
@traceable
def target_function(inputs):
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": inputs["question"]}
        ]
    )

    return {'answer': response.choices[0].message.content}

# Calling the decorated function creates a visible traced run.
test_result = target_function({'question': 'What is the speed of light?'})
print(test_result)

{'answer': 'The speed of light in a vacuum is approximately \\(299,792,458\\) meters per second (m/s). This value is often rounded to \\(3.00 \\times 10^8\\) m/s for convenience in calculations.'}


## Part 2: Dataset Creation (60-90 minutes)